In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.4/926.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 42.2 MB/s eta 0:00:00


In [3]:
 pip install reformer_pytorch==1.4.4

  Preparing metadata (setup.py) ... done
  Created wheel for axial-positional-embedding: filename=axial_positional_embedding-0.2.1-py3-none-any.whl size=2887 sha256=43b21cadb8529c0a6bb375bf351e478996ddb344e33e5f4e22455b289176a635
  Stored in directory: /root/.cache/pip/wheels/b1/cb/39/7ce7ff2d2fd37cfe1fe7b3a3c43cf410632b2ad3b3f3986d73
Successfully built axial-positional-embedding


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
import matplotlib.pyplot as plt
import tensorflow as tf
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import os
np.random.seed(42)


In [5]:
import pandas as pd

datos = pd.read_csv('/content/drive/MyDrive/Tesis/Data/SNconsumptionFinal.csv')

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [6]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [7]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [8]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [9]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [10]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [11]:
datosNormalizados.shape

(52416, 5)

In [12]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


In [13]:
first_column = datosNormalizados.pop(datosNormalizados.columns[1])
datosNormalizados["consumption"] = first_column

In [14]:
datosNormalizados.head(10)

,temp,zone2,zone3,hour,consumption
date,,,,,
2017-01-01 00:00:00,-2.051356,-0.851669,0.033500,-1.660858,0.151445
2017-01-01 00:10:00,-2.074813,-0.211097,0.016502,-1.660858,-0.433079
2017-01-01 00:20:00,-2.091152,-0.283791,-0.055068,-1.660858,-0.527709
2017-01-01 00:30:00,-2.122213,-0.411186,-0.174054,-1.660858,-0.651648
2017-01-01 00:40:00,-2.154568,-0.507632,-0.244729,-1.660858,-0.774750
2017-01-01 00:50:00,-2.165569,-0.597600,-0.293039,-1.660858,-0.872729
2017-01-01 01:00:00,-2.199865,-0.681090,-0.321667,-1.516340,-0.958984
2017-01-01 01:10:00,-2.223323,-0.746587,-0.396816,-1.516340,-1.035189
2017-01-01 01:20:00,-2.193880,-0.832236,-0.463913,-1.516340,-1.127306


In [15]:
target = "consumption"
futuros = 24
pasados = 12

In [16]:
df = datosNormalizados.reset_index()
df['target'] = df[target].shift(-futuros)

df = df.dropna()

df.head(40)

,date,temp,zone2,zone3,hour,consumption,target
0,2017-01-01 00:00:00,-2.051356,-0.851669,0.033500,-1.660858,0.151445,-1.713505
1,2017-01-01 00:10:00,-2.074813,-0.211097,0.016502,-1.660858,-0.433079,-1.719367
2,2017-01-01 00:20:00,-2.091152,-0.283791,-0.055068,-1.660858,-0.527709,-1.712668
3,2017-01-01 00:30:00,-2.122213,-0.411186,-0.174054,-1.660858,-0.651648,-1.715180
4,2017-01-01 00:40:00,-2.154568,-0.507632,-0.244729,-1.660858,-0.774750,-1.717693
5,2017-01-01 00:50:00,-2.165569,-0.597600,-0.293039,-1.660858,-0.872729,-1.742815
6,2017-01-01 01:00:00,-2.199865,-0.681090,-0.321667,-1.516340,-0.958984,-1.710993
7,2017-01-01 01:10:00,-2.223323,-0.746587,-0.396816,-1.516340,-1.035189,-1.700107
8,2017-01-01 01:20:00,-2.193880,-0.832236,-0.463913,-1.516340,-1.127306,-1.699269
9,2017-01-01 01:30:00,-2.224132,-0.889096,-0.499698,-1.516340,-1.195976,-1.675821


Se vuelve a dividir el dataframe

In [17]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(df, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36674, 7)
Las dimensiones de test son:  (10531, 7)
Las dimensiones de val son:  (5187, 7)


Se crea un dataloader modificado

In [18]:
!git clone https://github.com/thuml/Time-Series-Library.git

Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 1840, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 1840 (delta 3), reused 1 (delta 0), pack-reused 1828 (from 1)
Receiving objects: 100% (1840/1840), 78.20 MiB | 13.32 MiB/s, done.
Resolving deltas: 100% (1267/1267), done.


In [19]:
import sys
sys.path.append('Time-Series-Library')

In [20]:
from models import Crossformer
from utils.timefeatures import time_features

/content/Time-Series-Library/models/Crossformer.py:43: SyntaxWarning: "is" with a literal. Did you mean "=="?
  scale_block(configs, 1 if l is 0 else self.win_size, configs.d_model, configs.n_heads, configs.d_ff,
/content/Time-Series-Library/models/Crossformer.py:45: SyntaxWarning: "is" with a literal. Did you mean "=="?
  self.in_seg_num if l is 0 else ceil(self.in_seg_num / self.win_size ** l), configs.factor


In [21]:
from torch.utils.data import Dataset


class datasetCustom(Dataset):
    def __init__(self, flag='train', size=None,
                 features='MS', target='OT', scale=False, timeenc=1, freq='h', seasonal_patterns=None, dataProc=None):
        # size [seq_len, label_len, pred_len]
        #self.args = args
        # info
        self.seq_len = size[0]
        self.label_len = size[1]
        self.pred_len = size[2]
        # init
        assert flag in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}

        self.set_type = type_map[flag]

        self.features = features
        self.target = target
        #self.scale = scale
        self.timeenc = timeenc
        self.freq = freq

        #self.root_path = root_path
        self.dataProc = dataProc
        self.__read_data__()

    def __read_data__(self):
        #self.scaler = StandardScaler()
        df_raw = self.dataProc

        '''
        df_raw.columns: ['date', ...(other features), target feature]
        '''

        self.targetSet = df_raw[df_raw.columns[1:]].values
        cols = list(df_raw.columns)
        cols.remove("target")
        cols.remove('date')

        df_raw = df_raw[['date'] + cols] #+ ["target"]]

        '''
        num_train = int(len(df_raw) * 0.7)
        num_test = int(len(df_raw) * 0.2)
        num_vali = len(df_raw) - num_train - num_test
        border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]
        border2s = [num_train, num_train + num_vali, len(df_raw)]
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]
        '''
        if self.features == 'M' or self.features == 'MS':
            cols_data = df_raw.columns[1:]
            df_data = df_raw[cols_data]
        elif self.features == 'S':
            df_data = df_raw[[self.target]]
        '''
        if self.scale:
            train_data = df_data[border1s[0]:border2s[0]]
            self.scaler.fit(train_data.values)
            data = self.scaler.transform(df_data.values)
        else:'''
        data = df_data.values

        df_stamp = df_raw[['date']]  #[border1:border2]
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        #print(type(df_stamp))
        if self.timeenc == 0:
            df_stamp['year'] = df_stamp.date.apply(lambda row: row.year, 1)
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
            df_stamp['minute'] = df_stamp.date.apply(lambda row: row.minute, 1)
            data_stamp = df_stamp.drop(['date'], axis=1).values
        elif self.timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)
            data_stamp = data_stamp.transpose(1, 0)

        self.data_x = data
        self.data_y = data
        #print("dimensiones de x:", self.data_x.shape)
        #print("dimensiones de y:", self.data_y.shape)
        '''
        if self.set_type == 0 and self.args.augmentation_ratio > 0:
            self.data_x, self.data_y, augmentation_tags = run_augmentation_single(self.data_x, self.data_y, self.args)
        '''
        self.data_stamp = data_stamp

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end - self.label_len
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        #print("dimensiones de x en getItem:", seq_x.shape)
        #print("dimensiones de y en getItem:", seq_y.shape)
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]
        seq_target = self.targetSet[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark, seq_target

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1


In [22]:
features = 'MS'
target = 'consumption'
freq = '10T'

In [23]:
seq_len = 12 # input sequence length of Informer encoder
label_len = 12 # start token length of Informer decoder
pred_len = 1 # prediction sequence length
timeenc=0

In [24]:
from torch.utils.data import DataLoader

trainDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=train
        )
testDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=test
        )
valDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=val
        )

<ipython-input-21-7ee4e1730ba3>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_stamp['date'] = pd.to_datetime(df_stamp.date)
<ipython-input-21-7ee4e1730ba3>:70: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  df_stamp['year'] = df_stamp.date.apply(lambda row: row.year, 1)
<ipython-input-21-7ee4e1730ba3>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_stamp['year'] = 

In [25]:
a = trainDataset.data_y
a

array([[-2.05135573, -0.85166941,  0.03349964, -1.66085756,  0.15144479],
       [-2.07481311, -0.21109699,  0.01650173, -1.66085756, -0.43307937],
       [-2.0911524 , -0.28379117, -0.05506842, -1.66085756, -0.52770864],
       ...,
       [ 1.23607553,  0.5720641 , -0.81227818,  0.50690405,  0.48440731],
       [ 1.2829903 ,  0.5203727 , -0.82320012,  0.65142149,  0.4176651 ],
       [ 1.24578203,  0.51446511, -0.80135624,  0.65142149,  0.39219768]])

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl

class InformerLightning(pl.LightningModule):
    def __init__(self, model, args):
        super(InformerLightning, self).__init__()
        self.model = model  # El modelo de Informer
        self.args = args  # Argumentos como epochs

    def forward(self, batch_x, batch_x_mark, dec_inp, batch_y_mark):
        return self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)

    def training_step(self, batch, batch_idx):
        batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target = batch
        batch_x = batch_x.float()
        batch_y = batch_y.float()
        batch_x_mark = batch_x_mark.float()
        batch_y_mark = batch_y_mark.float()
        seq_target = seq_target.float()

        # Input del decoder
        dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()
        dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float()

        # Forward pass
        outputs = self(batch_x, batch_x_mark, dec_inp, batch_y_mark)
        #print("shape de output: ", outputs.shape)
        f_dim = -1 if self.args.features == 'MS' else 0
        outputs = outputs[:, -self.args.pred_len, f_dim:]
        real = seq_target[:, -self.args.pred_len-1, f_dim:]
        loss = nn.MSELoss()(real, outputs)

        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target = batch
        batch_x = batch_x.float()
        batch_y = batch_y.float()
        batch_x_mark = batch_x_mark.float()
        batch_y_mark = batch_y_mark.float()
        seq_target = seq_target.float()

        # Input del decoder
        dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()
        dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float()

        # Forward pass
        outputs = self(batch_x, batch_x_mark, dec_inp, batch_y_mark)
        f_dim = -1 if self.args.features == 'MS' else 0
        outputs = outputs[:, -self.args.pred_len, f_dim:]
        real = seq_target[:, -self.args.pred_len-1, f_dim:]
        loss = nn.MSELoss()(real, outputs)

        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        optimizer = self._select_optimizer()
        return optimizer

    def _select_optimizer(self):
        return optim.Adam(self.parameters(), lr=float(self.args.learning_rate))

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path, model, conf, *args, **kwargs):
        # Cargar el estado del modelo desde el checkpoint
        checkpoint = torch.load(checkpoint_path)
        # Crear una instancia del modelo
        instance = cls(model, args=conf, *args, **kwargs)
        instance.load_state_dict(checkpoint['state_dict'])
        return instance


In [27]:
entradas = 5
frecuencia = '10T'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [28]:
space = {
    'heads': hp.choice('heads', [2 ** i for i in range(1, 4)]),
    'e_layers': hp.quniform('e_layers', 1, 4, 1),
    'dropout':hp.quniform('dropout', 0, 0.5, 0.1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)]),
    'd_model': hp.choice('d_model', [2 ** i for i in range(6, 10)]),
    'factor': hp.choice('factor', [2, 3, 4]),
}

In [29]:
class Configs:
    def __init__(self, hyperparams):
        # Parámetros generales
        self.task_name = 'long_term_forecast'
        self.enc_in = entradas
        #self.dec_in = entradas
        #self.c_out = 1
        self.seq_len = pasados  # entrada
        self.label_len = label_len  # Definir este valor antes
        self.pred_len = 1  # Longitud de la predicción

        # Hiperparámetros que vienen del espacio de búsqueda
        self.d_model = int(hyperparams['d_model'])
        self.d_ff = self.d_model * 4
        self.n_heads = int(hyperparams['heads'])
        self.e_layers = int(hyperparams['e_layers'])
        self.factor = int(hyperparams['factor'])
        self.dropout = float(hyperparams['dropout'])
        self.learning_rate = float(hyperparams['learning_rate'])

        # Parámetros fijos o ajustables
        self.features = "MS"
        self.task_name = 'long_term_forecast'


In [30]:
def objective_fixed(params):
    loss_particular = 0.1783526688814163 # Loss fijo que deseas usar
    return {'loss': loss_particular, 'status': STATUS_OK}




In [31]:
def objective(params):
    print(params)
    early_stop_callback = EarlyStopping(
      monitor='val_loss',
      min_delta=0.00,
      patience=15,
      verbose=True,
      mode='min'
    )

    checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='min',
    )

    configs = Configs(params)


    model = Crossformer.Model(configs)
    informerModel = InformerLightning(model, configs)


    train_loader = DataLoader(trainDataset, batch_size=int(params["batch"]), shuffle=False)
    val_loader = DataLoader(testDataset, batch_size=int(params["batch"]), shuffle=False)


    trainer = pl.Trainer(
        callbacks=[early_stop_callback, checkpoint_callback],
        max_epochs=128,
        accelerator='gpu',
        enable_progress_bar=True,
        logger=False
    )


    trainer.fit(informerModel, train_loader, val_dataloaders=val_loader)

    bestModel = InformerLightning.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, model=model, conf=configs)

    informerModel.eval()
    preds = []
    trues = []

    test_loader = DataLoader(valDataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for i, (batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target) in enumerate(test_loader):
            batch_x = batch_x.float()
            batch_y = batch_y.float()
            batch_x_mark = batch_x_mark.float()
            batch_y_mark = batch_y_mark.float()
            seq_target = seq_target.float()


            dec_inp = torch.zeros_like(batch_y[:, -configs.pred_len:, :]).float()
            dec_inp = torch.cat([batch_y[:, :configs.label_len, :], dec_inp], dim=1).float().to(device)

            outputs = bestModel(batch_x, batch_x_mark, dec_inp, batch_y_mark)
            outputs = outputs[:, -configs.pred_len, -1:]
            batch_y = seq_target[:, -configs.pred_len-1, -1:]


            preds.append(outputs)
            trues.append(batch_y)

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    preds = preds.reshape(-1, preds.shape[-1])
    trues = trues.reshape(-1, trues.shape[-1])

    print("preds shape:", preds.shape)
    print("trues shape:", trues.shape)
    loss = mean_squared_error(trues, preds)

    return {'loss': loss, 'status': STATUS_OK}

In [32]:
import pytorch_lightning as pl

# Fijar la semilla para la reproducibilidad
pl.seed_everything(42)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

In [33]:
def combined_objective(params):
    if len(trials) < 9:
        return objective_fixed(params)  # Usar el loss particular
    else:
        return objective(params)   # Usar la función real

In [ ]:
#{'batch': 256, 'd_model': 128, 'dropout': 0.4, 'e_layers': 2.0, 'factor': 2, 'heads': 4, 'learning_rate': 0.0010445023972192214}
#0.1783526688814163

In [34]:

trials = Trials()

# Ejecutar la búsqueda de hiperparámetros
best = fmin(fn=combined_objective,
            space=space,
            algo=tpe.suggest,
            max_evals=12,
            trials=trials,
            rstate=np.random.default_rng(42))

print(best)

{'batch': 256, 'd_model': 512, 'dropout': 0.30000000000000004, 'e_layers': 1.0, 'factor': 4, 'heads': 4, 'learning_rate': 0.0007819763431394551}
 67%|██████▋   | 8/12 [00:00<00:00, 230.38trial/s, best loss: 0.1783526688814163]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type  | Params | Mode 
----------------------------------------
0 | model | Model | 25.2 M | train
----------------------------------------
25.2 M    Trainable params
0         Non-trainable params
25.2 M    Total params
100.999   Total estimated model params size (MB)
150       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 15 records. Best score: 0.517. Signaling Trainer to stop.
<ipython-input-26-051e6e8a8c6d>:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an is

preds shape:
(5175, 1)
trues shape:
(5175, 1)
100%|██████████| 12/12 [1:30:18<00:00, 451.53s/trial, best loss: 0.1783526688814163]
{'batch': 4, 'd_model': 1, 'dropout': 0.30000000000000004, 'e_layers': 4.0, 'factor': 1, 'heads': 1, 'learning_rate': 0.0017973043712331907}


In [35]:
print(best)

{'batch': 4, 'd_model': 1, 'dropout': 0.30000000000000004, 'e_layers': 4.0, 'factor': 1, 'heads': 1, 'learning_rate': 0.0017973043712331907}


In [ ]:

trials = Trials()

# Ejecutar la búsqueda de hiperparámetros
best = fmin(fn=combined_objective,
            space=space,
            algo=tpe.suggest,
            max_evals=12,
            trials=trials,
            rstate=np.random.default_rng(42))

print(best)

{'batch': 8, 'd_model': 64, 'dropout': 0.2, 'e_layers': 3.0, 'factor': 4, 'heads': 8, 'learning_rate': 1.8077878085009115e-06}
 58%|█████▊    | 7/12 [00:00<00:00, 176.06trial/s, best loss: 0.1783526688814163]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type  | Params | Mode 
----------------------------------------
0 | model | Model | 939 K  | train
----------------------------------------
939 K     Trainable params
0         Non-trainable params
939 K     Total params
3.758     Total estimated model params size (MB)
332       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.000 >= min_delta = 0.0. New best score: 0.212


Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
print(best)

In [ ]:

trials = Trials()

# Ejecutar la búsqueda de hiperparámetros
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=12,
            trials=trials,
            rstate=np.random.default_rng(42))

print(best)

{'batch': 128, 'd_model': 128, 'dropout': 0.30000000000000004, 'e_layers': 4.0, 'factor': 3, 'heads': 4, 'learning_rate': 0.0017973043712331907}
  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type  | Params | Mode 
----------------------------------------
0 | model | Model | 4.8 M  | train
----------------------------------------
4.8 M     Trainable params
0         Non-trainable params
4.8 M     Total params
19.088    Total estimated model params size (MB)
423       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.007 >= min_delta = 0.0. New best score: 0.242


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.016 >= min_delta = 0.0. New best score: 0.226


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.003 >= min_delta = 0.0. New best score: 0.223
